In [2]:
# Manual Stable Diffusion sampling that uses the scheduler.step update (fixed)
# Requires: diffusers, transformers, accelerate, safetensors, torch
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import numpy as np
import math
import os

# --- Config ---
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

prompt = "six cups"
height = 512
width = 512
num_inference_steps = 50
seed = 42
guidance_scale = 7.5
save_path = "six_cups.png"
do_cfg = True

torch.manual_seed(seed)

# --- Load pipeline and components ---
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, use_safetensors=True)
pipe = pipe.to(device)
pipe.safety_checker = None  # optional: skip NSFW checks if you want exact same output quickly

unet = pipe.unet
vae = pipe.vae
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
scheduler = pipe.scheduler

# eval mode
unet.eval()
vae.eval()
text_encoder.eval()

# --- 1) Encode prompt ---
text_inputs = tokenizer(
    prompt,
    padding="max_length",
    max_length=tokenizer.model_max_length,
    truncation=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    text_embeddings = text_encoder(text_inputs.input_ids)[0]   # shape [1, seq_len, hidden_dim]

# Classifier-free guidance: unconditional + conditional embeddings
if guidance_scale > 1.0:
    uncond_input = tokenizer(
        [""] * text_inputs.input_ids.shape[0],
        padding="max_length",
        max_length=tokenizer.model_max_length,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        uncond_embeddings = text_encoder(uncond_input.input_ids)[0]
    prompt_embeds_for_unet = torch.cat([uncond_embeddings, text_embeddings], dim=0)
else:
    prompt_embeds_for_unet = text_embeddings

# --- 2) Setup timesteps and latents ---
scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = scheduler.timesteps  # tensor of timesteps

# Prepare initial latents using scheduler.init_noise_sigma if available
# Many schedulers expose init_noise_sigma; fallback to 1.0 if not present
init_noise_sigma = getattr(scheduler, "init_noise_sigma", 1.0)

batch_size = 1
num_channels_latents = unet.in_channels
latent_height = height // pipe.vae_scale_factor
latent_width = width // pipe.vae_scale_factor

# Prepare latents (same as pipeline.prepare_latents)
latents = torch.randn(
    (batch_size, num_channels_latents, latent_height, latent_width),
    device=device,
    dtype=dtype,
) * init_noise_sigma

# If pipeline had pre-specified latents argument you could use that instead.

# Extra kwargs for scheduler.step (eta used for certain schedulers)
extra_step_kwargs = {}
if "eta" in scheduler.__class__.__name__.lower():
    extra_step_kwargs["eta"] = 0.0
# But better: use prepare_extra_step_kwargs from pipeline (internal helper) if available
try:
    extra_step_kwargs = pipe.prepare_extra_step_kwargs(generator=None, eta=0.0)
except Exception:
    # fallback: keep empty or minimal
    extra_step_kwargs = {}

# --- 3) Denoising loop (matching pipeline) ---
with torch.no_grad():
    for i, t in enumerate(timesteps):
        # Expand latents for classifier free guidance if needed
        latent_model_input = torch.cat([latents] * 2) if do_cfg else latents

        # Some schedulers require scaling the model input (e.g. LMSDiscreteScheduler)
        if hasattr(scheduler, "scale_model_input"):
            latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # Predict the noise residual with UNet
        # Use the correct forward signature: sample, timestep, encoder_hidden_states=...
        # Many UNet implementations accept `timestep_cond` etc; we keep minimal.
        unet_out = unet(
            latent_model_input,
            timestep=t,
            encoder_hidden_states=prompt_embeds_for_unet,
            return_dict=False,
        )
        # unet returns tuple (sample, ...) when return_dict=False
        noise_pred = unet_out[0]

        # Guidance
        if do_cfg:
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Use scheduler to compute previous sample
        # Many schedulers expect noise_pred, timestep, sample -> return_dict False -> [prev_sample]
        step_output = scheduler.step(noise_pred, t, latents, **extra_step_kwargs, return_dict=False)
        latents = step_output[0]  # updated latents

# --- 4) Decode latents to image ---
# stable-diffusion uses scaling factor in VAE config
latents = latents / pipe.vae.config.scaling_factor
with torch.no_grad():
    image = vae.decode(latents).sample

# postprocess: from [-1,1] to [0,255]
image = (image / 2 + 0.5).clamp(0, 1)
image_np = (image.cpu().permute(0, 2, 3, 1).numpy() * 255).round().astype(np.uint8)[0]
pil = Image.fromarray(image_np)
pil.save(save_path)
print("Saved to", save_path)


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  7.50it/s]
/tmp/ipykernel_3011181/1133354706.py:77: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  num_channels_latents = unet.in_channels


Saved to six_cups.png


In [7]:
# patchwise_conditioned_sd.py
import torch
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline
from PIL import Image
import numpy as np
import math
import os

# ---------- Config ----------
model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

main_prompt = "six cups on a wooden table, photorealistic"
patch_prompt = "one cup, photorealistic"
height = 512
width = 512
num_inference_steps = 50
seed = 42
guidance_scale_main = 7.5   # guidance for main prompt
guidance_scale_patch = 5.0  # guidance for patch-unets (maybe smaller)
save_path = "six_cups_patches.png"
do_cfg = True

torch.manual_seed(seed)

# ---------- Load pipeline & components ----------
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, use_safetensors=True)
pipe = pipe.to(device)
pipe.safety_checker = None

unet = pipe.unet
vae = pipe.vae
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
scheduler = pipe.scheduler

unet.eval()
vae.eval()
text_encoder.eval()

# ---------- Prepare prompt embeddings (main & patch) ----------

def make_embeddings(prompt, guidance_scale, do_cfg=True):
    """
    Returns embeddings for UNet input: either concatenated [uncond, cond] if CFG is used,
    or just cond embeddings otherwise.
    """
    # encode conditional
    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        text_embeddings = text_encoder(text_inputs.input_ids)[0]  # [1, seq_len, hidden_dim]

    if do_cfg and guidance_scale > 1.0:
        # unconditional
        uncond_input = tokenizer(
            [""] * text_inputs.input_ids.shape[0],
            padding="max_length",
            max_length=tokenizer.model_max_length,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            uncond_embeddings = text_encoder(uncond_input.input_ids)[0]

        # concat for classifier-free guidance
        prompt_embeds_for_unet = torch.cat([uncond_embeddings, text_embeddings], dim=0)
    else:
        prompt_embeds_for_unet = text_embeddings

    return prompt_embeds_for_unet

# main prompt embeddings (six cups)
main_emb_for_unet = make_embeddings(main_prompt, guidance_scale_main, do_cfg=do_cfg)

# patch prompt embeddings (one cup)
patch_emb_for_unet = make_embeddings(patch_prompt, guidance_scale_patch, do_cfg=do_cfg)


# ---------- Timesteps, latents ----------
scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = scheduler.timesteps

# init noise sigma & latents
init_noise_sigma = getattr(scheduler, "init_noise_sigma", 1.0)
batch_size = 1
num_channels_latents = unet.in_channels
latent_h = height // pipe.vae_scale_factor
latent_w = width // pipe.vae_scale_factor

latents = torch.randn((batch_size, num_channels_latents, latent_h, latent_w), device=device, dtype=dtype) * init_noise_sigma

# extra kwargs for scheduler.step (kept for compatibility)
try:
    extra_step_kwargs = pipe.prepare_extra_step_kwargs(generator=None, eta=0.0)
except Exception:
    extra_step_kwargs = {}

# ---------- Define patch layout (image-space) ----------
# Choose 6 disjoint image-space rectangles (x0, y0, x1, y1) in pixel coords (0..width, 0..height).
# Here is an example layout (two rows of three patches).
# You can change sizes/positions as needed.
patches_image = [
    (32,  80, 160, 240),   # patch 1
    (176, 80, 304, 240),   # patch 2
    (320, 80, 448, 240),   # patch 3
    (32,  272,160, 432),   # patch 4
    (176, 272,304, 432),   # patch 5
    (320, 272,448, 432),   # patch 6
]

# Convert to latent-space patch coordinates (integer indices)
def image_to_latent_coords(box, image_size=(width, height), latent_size=(latent_w, latent_h)):
    x0, y0, x1, y1 = box
    img_w, img_h = image_size
    l_w, l_h = latent_size
    # scale coordinates proportionally
    lx0 = int(round(x0 * l_w / img_w))
    ly0 = int(round(y0 * l_h / img_h))
    lx1 = int(round(x1 * l_w / img_w))
    ly1 = int(round(y1 * l_h / img_h))
    # clamp
    lx0 = max(0, min(lx0, l_w-1))
    lx1 = max(0, min(max(lx1, lx0+1), l_w))
    ly0 = max(0, min(ly0, l_h-1))
    ly1 = max(0, min(max(ly1, ly0+1), l_h))
    return (lx0, ly0, lx1, ly1)

patches_latent = [image_to_latent_coords(b, (width, height), (latent_w, latent_h)) for b in patches_image]

print("Latent patches:", patches_latent)

# ---------- Main denoising loop with patch-guidance ----------
with torch.no_grad():
    from tqdm import tqdm
    for i, t in tqdm(enumerate(timesteps-2)):
        # 1) Predict global noise residual (main prompt)
        latent_model_input = torch.cat([latents] * 2) if do_cfg else latents
        if hasattr(scheduler, "scale_model_input"):
            latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # UNet forward for main prompt
        unet_out = unet(latent_model_input, timestep=t, encoder_hidden_states=main_emb_for_unet, return_dict=False)
        noise_pred_main = unet_out[0]
        if do_cfg:
            n_uncond, n_text = noise_pred_main.chunk(2)
            noise_pred_main = n_uncond + guidance_scale_main * (n_text - n_uncond)

        # compute main latents_{t-1} using scheduler.step
        step_out = scheduler.step(noise_pred_main, t, latents, **extra_step_kwargs, return_dict=False)
        latents_prev = step_out[0].to(dtype=dtype)

        # 2) For each patch: extract, upsample to full latent size, run UNet with "one cup", step, downsample, add to latents_prev
        for (lx0, ly0, lx1, ly1) in patches_latent:
            # extract patch from current latents (at time t)
            patch = latents[:, :, ly0:ly1, lx0:lx1]  # shape [B, C, ph, pw]
            ph = patch.shape[2]
            pw = patch.shape[3]
            if ph <= 0 or pw <= 0:
                continue

            # upsample patch to full UNet spatial size (latent_h x latent_w)
            # use bilinear interpolation (patch is small so align_corners False)
            patch_upsampled = F.interpolate(patch, size=(latent_h, latent_w), mode="bilinear", align_corners=False)

            # UNet for patch prompt
            patch_input = torch.cat([patch_upsampled] * 2) if do_cfg else patch_upsampled
            if hasattr(scheduler, "scale_model_input"):
                patch_input_model = scheduler.scale_model_input(patch_input, t)
            else:
                patch_input_model = patch_input

            unet_out_patch = unet(patch_input_model, timestep=t, encoder_hidden_states=patch_emb_for_unet, return_dict=False)
            noise_pred_patch = unet_out_patch[0]
            if do_cfg:
                p_uncond, p_text = noise_pred_patch.chunk(2)
                noise_pred_patch = p_uncond + guidance_scale_patch * (p_text - p_uncond)

            # step this patch latent using scheduler (treating the upsampled patch as a full-latent sample)
            step_out_patch = scheduler.step(noise_pred_patch, t, patch_upsampled, **extra_step_kwargs, return_dict=False)
            patch_prev_full = step_out_patch[0]  # shape [B, C, latent_h, latent_w], dtype same as latents

            # downsample patch_prev_full back to patch spatial size
            patch_prev_down = F.interpolate(patch_prev_full, size=(ph, pw), mode="bilinear", align_corners=False)

            # add the patch contribution into latents_prev at proper coords
            # optional: you might want to blend (weighted add) instead of direct addition to avoid overflow
            latents_prev[:, :, ly0:ly1, lx0:lx1] = latents_prev[:, :, ly0:ly1, lx0:lx1] + patch_prev_down

        # After processing all patches, set latents = latents_prev for next timestep
        latents = latents_prev

# ---------- Decode and save ----------
latents = latents / pipe.vae.config.scaling_factor
with torch.no_grad():
    image = vae.decode(latents).sample

image = (image / 2 + 0.5).clamp(0, 1)
image_np = (image.cpu().permute(0, 2, 3, 1).numpy() * 255).round().astype(np.uint8)[0]
pil = Image.fromarray(image_np)
pil.save(save_path)
print("Saved to", save_path)


Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00,  8.71it/s]
/tmp/ipykernel_3011181/3197175842.py:94: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  num_channels_latents = unet.in_channels


Latent patches: [(4, 10, 20, 30), (22, 10, 38, 30), (40, 10, 56, 30), (4, 34, 20, 54), (22, 34, 38, 54), (40, 34, 56, 54)]


51it [00:21,  2.37it/s]

Saved to six_cups_patches.png



/tmp/ipykernel_3011181/3197175842.py:206: RuntimeWarning: invalid value encountered in cast
  image_np = (image.cpu().permute(0, 2, 3, 1).numpy() * 255).round().astype(np.uint8)[0]
